# Regression Methods — WPPSI-III IQ from Neonatal ECG/HRV

Template for regressing WPPSI-III IQ scores (QIT Full-Scale, QIV Verbal, QIP Performance, …) from neonatal linear + nonlinear HRV features (AuBE cohort, ~189 children, M0 recordings only).

Set ``TARGET_COLUMN`` below to the IQ you want to predict. To compare all IQs at once, use ``model_evaluation.py`` at the project root.

This notebook has been tested in Google Colab and VSCode. It is designed to use the `.env` file in this directory to locate the data files. Make sure to set it up by following the `.env.example` file.

**Research only — not a diagnostic tool.** Health data lives in `data/` and must **never** be committed or shared with AI tools.

See `Group A/dataset.md` for the full data dictionary and merge rules.

### Load data & Initialization

#### imports

In [ ]:
# --------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
)

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

# Pipeline & joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV, KFold, RepeatedKFold, RandomizedSearchCV 
import joblib

# Plotting style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

#### Google colab setup from vscode using colabsync

make sure to connect the colab sync

In [ ]:


#loading .env file for notebooks, it looks for .env file in the current working directory
import os
os.chdir("/content")
print("Current working directory: ", os.getcwd())
#list all files in the current working directory
print("Files in current working directory: ", os.listdir(os.getcwd()))

#change the current working directory to the directory of the notebook
os.chdir("/content/international-project/FSIQ-from-ECG/GroupA/Mateo")

print("Current working directory: ", os.getcwd())
print("Files in current working directory: ", os.listdir(os.getcwd()))

%load_ext dotenv
%reload_ext dotenv
%dotenv

#reset environment variables to avoid conflicts
os.environ.pop("DATA_FILE", None)
%dotenv
#create folder for models if it doesn't exist
models_path = os.getenv("MODELS_PATH")
if not os.path.exists(models_path):
    os.makedirs(models_path)
    print(f"Created models directory: {models_path}")

#### loading data from .env file

In [ ]:
data_file = os.getenv("DATA_FILE")
non_linearity_file = os.getenv("NON_LINEARITY_FILE")

def load_data(file_path):
    try:
        print(f"Loading data from {file_path}...")
        df = pd.read_csv(file_path, sep="\t", encoding="utf-8-sig", header=None) #change the delimiter to your needs.
        df = df.replace({ #fixing accents, the original file has some encoding issues, so we replace the incorrect characters with the correct ones, we cannot recover automatically
                    "Num�ro": "Numéro",
                    "Diab�te": "Diabète",
                    "Pr�matur�": "Prématuré",
                })
        
        column_names = df.iloc[0].tolist() #get the first row as column names
        descriptions = df.iloc[1].fillna("").tolist() #get the second row as descriptions
        descriptions = ["","",""] + descriptions #add empty strings for the first three columns, because they don't have descriptions and they don't have any tabs
        descriptions = descriptions[:len(column_names)] # truncate descriptions to match the number of columns
        df = df.iloc[2:].copy() # remove the first two rows
        df.columns = pd.MultiIndex.from_arrays([column_names, descriptions]) # set the first two rows as column names
        df = df.reset_index(drop=True) # reset the index

        return df
    except Exception as e:
        print(f"Error loading data from {file_path}: {e}")
        return None

df_linear = load_data(data_file)
df_non_linearity = load_data(non_linearity_file)


display(df_linear.head())
display(df_non_linearity.head())

#### loading results

In [ ]:
results_file = os.getenv("RESULTS_FILE")


def load_results(file_path):
    try:
        print(f"Loading results from {file_path}...")
        df_results = pd.read_csv(file_path, delimiter="\t", header=None) #change the delimiter to your needs.
        group_headers = df_results.iloc[0].fillna("") #get the first row as group headers
        group_headers = (
            group_headers.replace("", pd.NA)
            .ffill()
            .fillna("")
        )
        column_names = df_results.iloc[1].tolist() #get the second row as column names
        real_df_results = df_results.iloc[2:].copy() # remove the first two rows
        real_df_results.columns = pd.MultiIndex.from_arrays([group_headers, column_names]) #make the first two rows as column names
        real_df_results = real_df_results.reset_index(drop=True) # reset the index
        return real_df_results
    except Exception as e:
        print(f"Error loading results from {file_path}: {e}")
        return None 

print("results_file: ", results_file)

df_results = load_results(results_file)

display(df_results.head(10))

### Preprocessing

#### Target variable

We have to choose between those 4 variables:

- QIT (Quantitative Intelligence Test)
- QIV (Quantitative Intelligence Verbal)
- QIP (Quantitative Intelligence Performance)
- NCL (Nonverbal Cognitive Level)

For this analysis, we will use QIT as the target variable.

In [ ]:
TARGET_VARIABLE = "QIT"
AFTER_MERGE_TARGET_VARIABLE = "Note composite_(QIT)"

#### M0 Filterting

In [ ]:
passage_positions = [ #this is the position of the column that contains the passage information, we will use it to filter the data
    i for i, column in enumerate(df_linear.columns)
    if column[0] == "Passage"
] 

print(df_linear.columns[passage_positions])#MultiIndex([('Passage', '-')],)

passage = (
    df_linear.iloc[:, passage_positions[0]]
    .astype("string")
    .str.strip()
) #this is the column that contains the passage information, we will use it to filter the data

df_linear_m0 = df_linear.loc[passage.eq("M0")].copy() #display the rows where the passage is M0, we will use this dataframe for the analysis

display(df_linear_m0.head())
print(df_linear_m0.shape)

df_non_linearity_m0 = df_non_linearity.loc[passage.eq("M0")].copy()
display(df_non_linearity_m0.head())
print(df_non_linearity_m0.shape)

#### Remove rows of WSSPI with missing values in the target variable

In [ ]:
df_results_clean = df_results.dropna(subset=[("Note composite", TARGET_VARIABLE)]).copy() #remove rows with missing values in the target variable

print(df_results_clean.shape)
print(df_results.shape)

#### Merge linear + nonlinear HRV on Monogramme + DDN + Passage

In [ ]:
#convert multiindex to single index for easier merging
df_linear_m0_clean = df_linear_m0.copy()
df_linear_m0_clean.columns = [str(col[0]) + "_(" + str(col[1]) + ")" if col[1] != "" else str(col[0]) for col in df_linear_m0_clean.columns]

display(df_linear_m0_clean.head())
#print every single column name in the dataframe, not truncated
print(df_linear_m0_clean.columns.to_list())

df_non_linearity_m0_clean = df_non_linearity_m0.copy()
df_non_linearity_m0_clean.columns = [str(col[0]) + "_(" + str(col[1]) + ")" if col[1] != "" else str(col[0]) for col in df_non_linearity_m0_clean.columns]

#merge
df_merged = pd.merge(
    df_linear_m0_clean,
    df_non_linearity_m0_clean,
    on=["Monogramme", "DDN", "Passage_(-)"],
    how="inner",
)

display(df_merged.head())

#### Merge HRV features with WSSPI data on Monogramme + DDN

In [ ]:
#list of columns in df_results_clean
df_results_clean_tmp = df_results_clean.copy()
#df_results_clean_tmp.columns
print(df_results_clean_tmp.columns.to_list())

#for df_results, the monogramme is ('', 'N°'), the DDN is ('', 'DDN'), 
#convert multiindex to single index for easier merging
df_results_clean_tmp.columns = [str(col[0]) + "_(" + str(col[1]) + ")" if col[1] != "" else str(col[0]) for col in df_results_clean_tmp.columns]

print(df_results_clean_tmp.columns.to_list())

#edit column names in df_results_clean_tmp to match df_merged
df_results_clean_tmp = df_results_clean_tmp.rename(columns={
    "_(nan)": "Monogramme",
    "_(DDN)": "DDN",
})

#merge df_merged with df_results_clean_tmp on Monogramme and DDN without certains columns from results_clean_tmp, we will keep only the columns that are needed for the analysis, which are Monogramme, DDN, and the target variable AFTER_MERGE_TARGET_VARIABLE
df_merged_full = pd.merge(
    df_merged,
    df_results_clean_tmp.loc[:, ["Monogramme", "DDN", AFTER_MERGE_TARGET_VARIABLE]],
    on=["Monogramme", "DDN"],
    how="inner",
)

display(df_merged_full.head())
print(df_merged_full.shape)
print(df_merged_full.columns.to_list())
    

#### Drop fully empty feature columns & impute missing values with mean


In [ ]:
#drop fully empty feature columns & impute missing values with mean

def drop_empty(df):
    #drop fully empty feature columns 
    df = df.dropna(axis=1, how="all")
    #we know that nan can be represented as a "-" in the data, so we replace "-" with np.nan
    df = df.replace("-", np.nan)
    df = df.replace("", np.nan)
    df = df.replace(" ", np.nan)
    df = df.replace("nan", np.nan)
    df = df.replace("NaN", np.nan)
    df = df.dropna(axis=1, how="all") #drop fully empty feature columns again after replacing "-" with np.nan
    #impute missing values with column mean
    df = df.fillna(df.mean(numeric_only=True))
    return df

df_merged_full_clean = drop_empty(df_merged_full)

display(df_merged_full_clean.head())
print(df_merged_full_clean.shape)
print("missing values in df_merged_full_clean: ", df_merged_full_clean.isnull().sum().sum())

print('columns with how many missing values:')
df_missing = df_merged_full_clean.isnull().sum()
df_missing = df_missing[df_missing > 0]
print(df_missing.sort_values(ascending=False))

### Feature selection / engineering

#### Remove features with more than 80% missing values


In [ ]:
def remove_high_missing(df, threshold=0.8):
    #remove features with more than 80% missing values
    missing = df.isnull().sum() / len(df)
    to_remove = missing[missing > threshold].index
    list_removed = to_remove.tolist()
    print(f"Columns to remove (more than {threshold*100}% missing values): {
list_removed}")
    df = df.drop(columns=to_remove)
    print(f"Removed {len(to_remove)} columns with more than {threshold*100}% missing values.")
    
    print(f"Remaining columns: {df.shape[1]}")  
    return df

df_merged_full_clean_80 = remove_high_missing(df_merged_full_clean, threshold=0.8)

display(df_merged_full_clean_80.head())
print(df_merged_full_clean_80.shape)

print("missing values in df_merged_full_clean_80: ", df_merged_full_clean_80.isnull().sum().sum())

#### Correlation analysis & removal of highly correlated features (> 0.95)

In [ ]:
#list of columns types with how many missing values
print('columns types with how many missing values:')
df_missing_types = df_merged_full_clean_80.dtypes
df_missing_types = df_missing_types.value_counts()
print(df_missing_types)

#with the analysis above, we can see that columns are usually of type object, and only 5 columns are of type float64. A lot of columns are of type object even though they should be numeric. We will convert all columns to numeric, and if they cannot be converted, we will drop them.
df_merged_full_clean_80_num = df_merged_full_clean_80.apply(pd.to_numeric, errors='coerce')
#we gotta keep columns that are not numeric, so we will create a new dataframe with only the non-numeric columns by selecting the columns that are fully NaN after the conversion to numeric, and we will keep them in a separate dataframe for later use. We will also print the list of non-numeric columns.
df_merged_full_non_numeric = df_merged_full_clean_80_num.isnull().all()
df_merged_full_non_numeric = df_merged_full_clean_80.loc[:, df_merged_full_non_numeric]
non_numeric_columns = df_merged_full_non_numeric.columns
print("non-numeric columns: ", non_numeric_columns)
print("non-numeric columns types: ", df_merged_full_non_numeric.dtypes)

display(df_merged_full_non_numeric.head())

print("columns converted to numeric: ", df_merged_full_clean_80_num.dtypes.value_counts())

def correlation_analysis(df, threshold=0.95):
    #correlation analysis & removal of highly correlated features (> 0.95)
    numeric_df = df.select_dtypes(include=[np.number])
    corr_matrix = numeric_df.corr().abs()
    #plot matrix as a heatmap but only for the ones that are above the threshold, and only for the upper triangle of the matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", fmt=".2f", square=True, cbar_kws={"shrink": 0.8})
    plt.title("Correlation Matrix")
    plt.show()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    print(f"Columns to drop (correlation > {threshold}): {to_drop}")
    df = df.drop(columns=to_drop)
    print(f"Removed {len(to_drop)} columns with correlation > {threshold}.")
    print(f"Remaining columns: {df.shape[1]}")
    return df

df_merged_full_clean_80_corr = correlation_analysis(df_merged_full_clean_80_num, threshold=0.95)

df_merged_full_clean_80_corr[non_numeric_columns] = df_merged_full_non_numeric

display(df_merged_full_clean_80_corr.head())


#### Variance threshold (remove near-zero variance features)



In [ ]:
def remove_nearzero_variance(df, threshold=0.01):
    #remove near-zero variance features
    variance = df.var(numeric_only=True)
    to_remove = variance[variance < threshold].index
    list_removed = to_remove.tolist()
    print(f"Columns to remove (variance < {threshold}): {list_removed}")
    df = df.drop(columns=to_remove)
    print(f"Removed {len(to_remove)} columns with variance < {threshold}.")
    print(f"Remaining columns: {df.shape[1]}")
    return df


print("list of columns types:")
print(df_merged_full_clean_80_corr.dtypes.value_counts())
#good news, this has the right types

df_merged_full_clean_80_corr = remove_nearzero_variance(df_merged_full_clean_80_corr, threshold=0.01)

print("list of columns types after removing near-zero variance features:")
print(df_merged_full_clean_80_corr.dtypes.value_counts())


#### Lasso/ElasticNet feature selection coefficient screening

this function will perform lasso/elasticnet feature selection and return the selected features. It will also print the ranking of features by absolute value of coefficients and plot a barplot of the coefficients.

What a feature has to have to be selected is that its coefficient has to be non-zero. The function will also print the number of selected features.

In [ ]:
def lasso_elasticnet_feature_selection(df, target_variable, alpha=0.01, l1_ratio=0.5):
    #lasso/elasticnet feature selection coefficient screening
    X = df.drop(columns=[target_variable])
    y = df[target_variable]
    
    #impute missing values with mean for numeric columns
    X = X.fillna(X.mean(numeric_only=True))
    
    #standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.select_dtypes(include=[np.number]))
    
    #fit lasso model
    lasso = Lasso(alpha=alpha)
    lasso.fit(X_scaled, y)
    
    #get coefficients
    coef = pd.Series(lasso.coef_, index=X.select_dtypes(include=[np.number]).columns)
    
    #select features with non-zero coefficients
    selected_features = coef[coef != 0].index.tolist()
    
    #barplot of coefficients and draw full horizontal red barplot for features that haven't been selected (i.e. have a coefficient of 0)
    plt.figure(figsize=(12, 12))
    coef.sort_values().plot(kind="barh")
    plt.title("Lasso Coefficients")
    plt.xlabel("Coefficient Value")
    plt.ylabel("Feature")
    for position, feature in enumerate(coef.sort_values().index):
        if coef.sort_values()[feature] == 0:
            plt.axhline(
                y=position,
                color="red",
                linestyle="--",
                linewidth=2
            )
    plt.show()
    
    return selected_features

print("list of columns types before lasso/elasticnet feature selection:")
print(df_merged_full_clean_80_corr.dtypes.value_counts())

selected_features = lasso_elasticnet_feature_selection(df_merged_full_clean_80_corr, AFTER_MERGE_TARGET_VARIABLE, alpha=0.01, l1_ratio=0.5)

print("list of selected features after lasso/elasticnet feature selection:")
print(selected_features)

print("Number of selected features:")
print(len(selected_features))

df_merged_lasso_selected = df_merged_full_clean_80_corr[selected_features + [AFTER_MERGE_TARGET_VARIABLE]]

display(df_merged_lasso_selected.head())

print(df_merged_lasso_selected.shape)

#### Taking off unrelated features 

In [ ]:
df_took_off = df_merged_lasso_selected.drop(columns=["Numéro_x"])

#taking off all note_composite columns except for the target variable AFTER_MERGE_TARGET_VARIABLE
columns_to_drop = [col for col in df_took_off.columns if "Note composite" in col and col != AFTER_MERGE_TARGET_VARIABLE]
df_took_off = df_took_off.drop(columns=columns_to_drop)

display(df_took_off.head())


#### Standardization of features on training data

In [ ]:
def std_feature_traindata(X_train):
    #standardize features on training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    return X_train_scaled, scaler

df_merged_lasso_selected_numeric = df_took_off.select_dtypes(include=[np.number])
df_merged_lasso_selected_non_numeric = df_took_off.select_dtypes(exclude=[np.number])

target_variable = AFTER_MERGE_TARGET_VARIABLE
try:
    X = df_merged_lasso_selected_numeric.drop(columns=[target_variable])
    y = df_merged_lasso_selected_numeric[target_variable]
    X_train_scaled, scaler = std_feature_traindata(X)
except KeyError as e:
    print(f"Error: {e}")
    
    
nan_counts = X.isna().sum()
#print count and their types
print("Count of NaN values in each column:\n", nan_counts[nan_counts > 0])
print("Count of NaN values in each column types:\n", X.dtypes[nan_counts > 0])




#### Impute missing values with mean on training data


In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean') #impute missing values with mean for numeric columns
X_train_imputed = imputer.fit_transform(X_train_scaled)

print("nan_counts after imputation: ", np.isnan(X_train_imputed).sum()  )

#### Dataset get-off filtering

In [ ]:
### take off the columns that are related to the WPPSI results

df_merged_lasso_selected_no_wppsi = df_merged_lasso_selected

#### replot correlation matrix after feature selection

In [ ]:
#we use X_train_imputed and y for plots, correlation ranking with y

write_path_plots = os.getenv("WRITE_PATH_PLOTS")
if not os.path.exists(write_path_plots):
    os.makedirs(write_path_plots)
    print(f"Created plots directory: {write_path_plots}")
plt.figure(figsize=(10, 15))
#compute correlation matrix with y
sns.heatmap(df_took_off.corr()[[AFTER_MERGE_TARGET_VARIABLE]].sort_values(by=AFTER_MERGE_TARGET_VARIABLE, ascending=False), annot=True, cmap="coolwarm", fmt=".2f", square=False, cbar_kws={"shrink": 1.0})
plt.title("Correlation Matrix with target variable")
plt.savefig(os.path.join(write_path_plots, "correlation_matrix_with_target_variable.png"), dpi=300, bbox_inches='tight')
plt.show()

#save plot as png at path WRITE_PATH_PLOTS




#save correlation matrix as csv at path WRITE_PATH_PLOTS
corr_matrix = df_took_off.corr()[[AFTER_MERGE_TARGET_VARIABLE]]
corr_matrix.to_csv(os.path.join(write_path, "correlation_matrix_with_target_variable.csv"))

#### Save dataset in data folder for future use (locally, .gitignore)

In [ ]:
write_path = os.getenv("WRITE_PATH")

#save the dataframe to a csv file in the data folder for future use (locally, .gitignore)

X_train_imputed_df = pd.DataFrame(X_train_imputed, columns=X.columns)
X_and_Y = pd.concat([X_train_imputed_df, y.reset_index(drop=True)], axis=1)

#force replacement of the file if it already exists

if os.path.exists(os.path.join(write_path, "df_merged_lasso_selected_imputed.csv")):
    os.remove(os.path.join(write_path, "df_merged_lasso_selected_imputed.csv"))
X_and_Y.to_csv(os.path.join(write_path, "df_merged_lasso_selected_imputed.csv"), index=False)


### Regression!

#### ElasticNet regression with cross-validation

time with colab = 28s

In [ ]:
# parameters

#alpha is the regularization strength, must be a positive float. 

#alpha would be logspace from -4 to 1

alpha_values = np.logspace(-4, 1, 50)

l1_ratio_values = np.linspace(0, 1, 10) #[0.1, 0.3, 0.5, 0.7, 0.9]

models_path = os.getenv("MODELS_PATH")

ElasticNet is a linear regression model that combines L1 and L2 regularization. It is useful when there are multiple features that are correlated with each other. The model has two hyperparameters: alpha (the regularization strength) and l1_ratio (the mix between L1 and L2 regularization).

In [ ]:
# model
from sklearn.base import clone

from sklearn.pipeline import make_pipeline
model = ElasticNet()

#import data from the csv file we just saved, and split it into X and y

df_merged_lasso_selected = pd.read_csv(os.path.join(write_path, "df_merged_lasso_selected_imputed.csv"))
X_train = df_merged_lasso_selected.drop(columns=[target_variable])
y_train = df_merged_lasso_selected[target_variable]

from sklearn.model_selection import RandomizedSearchCV , KFold, RandomizedSearchCV

pipeline = make_pipeline(
    StandardScaler(),
    ElasticNet(max_iter=10000, random_state=42)
)

param_grid = {
    "elasticnet__alpha": alpha_values,
    "elasticnet__l1_ratio": l1_ratio_values
}

grid_search = RandomizedSearchCV(
    pipeline,
    param_grid,
    n_iter=50,  # Number of parameter settings that are sampled. n_iter trades off runtime vs quality of the solution.
    scoring="neg_mean_absolute_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(grid_search.best_params_)
print(-grid_search.best_score_)

joblib.dump(best_model, os.path.join(models_path, "best_elasticnet_model.joblib"))


#### Random Forest regression with cross-validation

time with colab = 12m 21s

In [ ]:
#parameters

n_estimators = [200, 500]
max_depth = [None, 3,5,7]
min_samples_split = [5,10,20]
max_features = ["sqrt", "log2"]


In [ ]:
model = RandomForestRegressor()

pipeline_rf = make_pipeline(
    StandardScaler(),
    RandomForestRegressor(random_state=42)
)

param_grid_rf = {
    "randomforestregressor__n_estimators": n_estimators,
    "randomforestregressor__max_depth": max_depth,
    "randomforestregressor__min_samples_split": min_samples_split,
    "randomforestregressor__max_features": max_features
}

grid_search_rf = RandomizedSearchCV (
    pipeline_rf,
    param_grid_rf,
    n_iter=50,  # Number of parameter settings that are sampled. n_iter trades off runtime vs quality of the solution.
    scoring="neg_mean_absolute_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=1 #n_jobs=1 means using all processors, you can change it to a specific number if you want to limit the number of processors used
)

grid_search_rf.fit(X_train, y_train)

best_model_rf = grid_search_rf.best_estimator_
print(grid_search_rf.best_params_)
print(-grid_search_rf.best_score_)

joblib.dump(best_model_rf, os.path.join(models_path, "best_randomforest_model.joblib"))


#### LightGBM regression with cross-validation

time with colab = 1m 37s

In [ ]:
#params
n_estimators = [100, 300]
learning_rate = [0.01, 0.05, 0.1]
max_depth = [3, 5, 7]
min_child_samples = [1,5]
alpha = [0.001, 0.1, 0.5, 1.0]
reg_lambda = [0.0, 0.1, 0.5, 1.0]

In [ ]:
#model training

from lightgbm import LGBMRegressor

model = LGBMRegressor(random_state=42, objective='regression', n_jobs=1, verbosity=-1)

pipeline_lgbm = make_pipeline(
    LGBMRegressor(random_state=42, objective='regression', n_jobs=1, verbosity=-1)
)

param_grid_lgbm = {
    "lgbmregressor__n_estimators": n_estimators,
    "lgbmregressor__learning_rate": learning_rate,
    "lgbmregressor__max_depth": max_depth,
    "lgbmregressor__min_child_samples": min_child_samples,
    "lgbmregressor__alpha": alpha, #alpha is the L1 regularization term on weights, must be a positive float. 
    "lgbmregressor__reg_lambda": reg_lambda #reg_lambda is the L2 regularization term on weights, must be a positive float. 
}

grid_search_lgbm = RandomizedSearchCV(
    pipeline_lgbm,
    param_grid_lgbm,
    n_iter=50,  # Number of parameter settings that are sampled. n_iter trades off runtime vs quality of the solution.
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=1
)

grid_search_lgbm.fit(X_train, y_train)

best_model_lgbm = grid_search_lgbm.best_estimator_
print(grid_search_lgbm.best_params_)
print(-grid_search_lgbm.best_score_)

joblib.dump(best_model_lgbm, os.path.join(models_path, "best_lightgbm_model.joblib"))

#### XGBoost regression with cross-validation
time with colab = 2min

In [ ]:
#params

n_estimators = [100, 300]
learning_rate = [0.01, 0.05, 0.1]
max_depth = [3, 5, 7]
min_child_weight = [1, 5]
subsample = [0.8, 1.0]
colsample_bytree = [0.8, 1.0]
reg_alpha = [0.0,0.1,1.0]
reg_lambda = [0.1,1.0,10.0]

In [ ]:
#model training

from xgboost import XGBRegressor

pipeline_xgb = make_pipeline(
    XGBRegressor(random_state=42, objective='reg:squarederror', n_jobs=1, verbosity=0, eval_metric='mae')
)

param_grid_xgb = {
    "xgbregressor__n_estimators": n_estimators,
    "xgbregressor__learning_rate": learning_rate,
    "xgbregressor__max_depth": max_depth,
    "xgbregressor__min_child_weight": min_child_weight,
    "xgbregressor__subsample": subsample,
    "xgbregressor__colsample_bytree": colsample_bytree,
    "xgbregressor__reg_alpha": reg_alpha,
    "xgbregressor__reg_lambda": reg_lambda,
}

grid_search_xgb = RandomizedSearchCV (
    pipeline_xgb,
    param_grid_xgb,
    n_iter=50,  # Number of parameter settings that are sampled. n_iter trades off runtime vs quality of the solution.
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=1
)

grid_search_xgb.fit(X_train, y_train)

best_model_xgb = grid_search_xgb.best_estimator_
print(grid_search_xgb.best_params_)
print(-grid_search_xgb.best_score_)

joblib.dump(best_model_xgb, os.path.join(models_path, "best_xgboost_model.joblib"))

#### Support Vector Regression (SVR) with cross-validation

time with colab = 31s

In [ ]:
#params

C = [0.1, 1, 10, 100]
epsilon = [0.01, 0.1, 1.0]
kernel = ['rbf', 'linear']
gamma = ['scale', 'auto']

In [ ]:
#model training

from sklearn.svm import SVR

pipeline_svr = make_pipeline(
    StandardScaler(),
    SVR()
)

param_grid_svr = {
    "svr__C": C,
    "svr__epsilon": epsilon,
    "svr__kernel": kernel,
    "svr__gamma": gamma
}

grid_search_svr = RandomizedSearchCV(
    pipeline_svr,
    param_grid_svr,
    n_iter=50,  # Number of parameter settings that are sampled. n_iter trades off runtime vs quality of the solution.
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=1,
    error_score='raise'
)

grid_search_svr.fit(X_train, y_train)

best_model_svr = grid_search_svr.best_estimator_
print(grid_search_svr.best_params_)
print(-grid_search_svr.best_score_)

joblib.dump(best_model_svr, os.path.join(models_path, "best_svr_model.joblib"))

#### Summary of results

All models were trained to predict the target variable using the preprocessed dataset. Hyperparameters were selected using 5-fold cross-validation, with mean absolute error (MAE) as the optimization metric. The final model was refitted using the best hyperparameter combination.

| Model | Preprocessing | Main hyperparameters tuned | Evaluation |
|---|---|---|---|
| ElasticNet | Median imputation + standardization | `alpha`, `l1_ratio` | 5-fold CV |
| LightGBM | Imputation; scaling not required | `n_estimators`, `learning_rate`, `max_depth`, `min_child_samples`, `reg_alpha`, `reg_lambda` | 5-fold CV |
| XGBoost | Imputation; scaling not required | `n_estimators`, `learning_rate`, `max_depth`, `min_child_weight`, `subsample`, `reg_alpha`, `reg_lambda` | 5-fold CV |
| SVR | Median imputation + standardization | `C`, `epsilon`, `kernel`, `gamma` | 5-fold CV |

The models were compared using MAE, MSE, RMSE, and \(R^2\). Lower MAE, MSE, and RMSE indicate better performance, whereas higher \(R^2\) indicates better explanatory performance.

The target variable was `Note composite_(QIT)`. Missing predictor values were imputed and selected features were used for modelling. Each model was evaluated using 5-fold cross-validation, with hyperparameters selected by minimizing mean absolute error (MAE).

| Model | Best hyperparameters | Cross-validated MAE |
|---|---|---:|
| ElasticNet | `alpha = 6.2506`, `l1_ratio = 0.6667` | **10.3853** |
| SVR | `C = 0.1`, `epsilon = 0.01`, `kernel = rbf`, `gamma = scale` | **10.4125** |
| XGBoost | `n_estimators = 100`, `learning_rate = 0.01`, `max_depth = 7`, `min_child_weight = 1`, `subsample = 0.8`, `colsample_bytree = 1.0`, `reg_alpha = 1.0`, `reg_lambda = 10.0` | **10.5920** |
| Random Forest | `n_estimators = 200`, `min_samples_split = 20`, `max_features = log2`, `max_depth = 3` | **10.6362** |
| LightGBM | `alpha = 0.001`, `learning_rate = 0.01`, `max_depth = 3`, `min_child_samples = 5`, `n_estimators = 300`, `reg_lambda = 1.0` | **11.1753** |

ElasticNet achieved the lowest cross-validated MAE (**10.3853**), followed by SVR (**10.4125**) and XGBoost (**10.5920**). Random Forest and LightGBM performed slightly worse, with MAEs of **10.6362** and **11.1753**, respectively. Therefore, ElasticNet was the best-performing model according to cross-validated MAE.

### Standerdized evaluation across all models and all IQ targets

In [ ]:


model_specs = {
    "ElasticNet": {
        "estimator": make_pipeline(
            StandardScaler(),
            ElasticNet(max_iter=10000, random_state=42)
        ),
        "params": {
            "elasticnet__alpha": np.logspace(-4, 1, 20),
            "elasticnet__l1_ratio": [0.1, 0.5, 0.9]
        }
    },

    "RandomForest": {
        "estimator": RandomForestRegressor(random_state=42, n_jobs=1),
        "params": {
            "n_estimators": [200, 500],
            "max_depth": [None, 3, 5, 7],
            "min_samples_split": [5, 10, 20],
            "max_features": ["sqrt", "log2"]
        }
    },

    "LightGBM": {
        "estimator": LGBMRegressor(
            objective="regression",
            random_state=42,
            n_jobs=1,
            verbosity=-1
        ),
        "params": {
            "n_estimators": [100, 300],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [3, 5, 7],
            "min_child_samples": [1, 5],
            "reg_alpha": [0.001, 0.1, 0.5],
            "reg_lambda": [0.0, 0.1, 1.0]
        }
    },

    "XGBoost": {
        "estimator": XGBRegressor(
            objective="reg:squarederror",
            random_state=42,
            n_jobs=1,
            eval_metric="mae"
        ),
        "params": {
            "n_estimators": [100, 300],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [3, 5, 7],
            "min_child_weight": [1, 5],
            "subsample": [0.8, 1.0],
            "reg_alpha": [0.0, 0.1, 1.0],
            "reg_lambda": [0.1, 1.0, 10.0]
        }
    },

    "SVR": {
        "estimator": make_pipeline(
            StandardScaler(),
            SVR()
        ),
        "params": {
            "svr__C": [0.1, 1, 10, 100],
            "svr__epsilon": [0.01, 0.1, 1.0],
            "svr__kernel": ["rbf", "linear"],
            "svr__gamma": ["scale", "auto"]
        }
    }
}

warning: time to execute this cell: 75min~

In [ ]:
#import pipeline
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedKFold,
    cross_validate
)
#cv strategy: 10-fold cross-validation with shuffle and random_state=42, 10 repeats, target is QIT (primary target), QIV,QIP,NCL
%dotenv

AFTER_MERGE_TARGET_VARIABLE = os.getenv("AFTER_MERGE_TARGET_VARIABLE")
write_path = os.getenv("WRITE_PATH")
df_main = pd.read_csv(os.path.join(write_path, "df_merged_lasso_selected_imputed.csv"))

X = df_main.drop(columns=[AFTER_MERGE_TARGET_VARIABLE])
y = df_main[[AFTER_MERGE_TARGET_VARIABLE]]

scoring_metrics = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}

# this would take a huge amount of time to run
# inner_cv = RepeatedKFold(n_splits=10, n_repeats=10, random_state=42)
# outer_cv = RepeatedKFold(n_splits=10, n_repeats=10, random_state=42)

inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)
outer_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

scoring = scoring_metrics

def nested_cv_evaluation(estimator, param_grid, X, y):
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_grid,
        n_iter=50,  # Number of parameter settings that are sampled. n_iter trades off runtime vs quality of the solution.
        scoring="neg_mean_absolute_error",
        cv=inner_cv,
        n_jobs=1,
        error_score="raise",
        random_state=42
    )

    scores = cross_validate(
        search,
        X,
        y,
        cv=outer_cv,
        scoring=scoring,
        n_jobs=1
    )

    mae = -scores["test_MAE"]
    mse = -scores["test_MSE"]
    rmse = -scores["test_RMSE"]
    r2 = scores["test_R2"]

    return {
        "MAE_mean": mae.mean(),
        "MAE_std": mae.std(),
        "MSE_mean": mse.mean(),
        "MSE_std": mse.std(),
        "RMSE_mean": rmse.mean(),
        "RMSE_std": rmse.std(),
        "R2_mean": r2.mean(),
        "R2_std": r2.std()
    }
    
    
results = {}

for model_name, spec in model_specs.items():
    print(f"Evaluating {model_name}...")
    results[model_name] = nested_cv_evaluation(spec["estimator"], spec["params"], X, y[AFTER_MERGE_TARGET_VARIABLE])
    print(f"Results for {model_name}: {results[model_name]}")
    
results_df = pd.DataFrame(results).T
results_df.to_csv(os.path.join(write_path, "model_evaluation_results.csv"), index=True
)
display(results_df)

### Nested cross-validation results

The models were evaluated using nested cross-validation. Results are reported as mean ± standard deviation across the outer folds.

| Model | MAE | MSE | RMSE | R² |
|---|---:|---:|---:|---:|
| ElasticNet | 10.440 ± 1.149 | 167.929 ± 40.140 | 12.864 ± 1.566 | −0.050 ± 0.050 |
| Random Forest | 10.595 ± 1.213 | 173.164 ± 40.331 | 13.065 ± 1.569 | −0.084 ± 0.055 |
| LightGBM | 10.677 ± 1.262 | 173.606 ± 39.493 | 13.086 ± 1.541 | −0.089 ± 0.076 |
| XGBoost | 10.597 ± 1.244 | 172.217 ± 40.302 | 13.027 ± 1.583 | −0.078 ± 0.085 |
| SVR | 10.446 ± 1.107 | 167.461 ± 37.390 | 12.857 ± 1.470 | −0.050 ± 0.047 |

### Interpretation

ElasticNet achieved the lowest MAE (10.440), while SVR achieved the lowest MSE (167.461) and RMSE (12.857). ElasticNet had the highest R² (−0.050), although it was almost identical to SVR (−0.050).

All models obtained negative R² values. This indicates that they performed worse than a baseline model that always predicts the mean target value. Therefore, none of the models demonstrates useful predictive performance on this evaluation. The differences between the models are small compared with their standard deviations, so there is no clear practical advantage among them. These results suggest that the current features and modelling approach do not adequately predict the target variable.

#### Plots

packages

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os


import seaborn as sns
import numpy as np

import os
os.chdir("/content/international-project/FSIQ-from-ECG/GroupA/Mateo")

#line magic function not found: run this line 
# %load_ext dotenv
%reload_ext dotenv
%dotenv

plot of MAE

In [ ]:
#bar plot of mae per model with error bars for std, sorted by mae mean

#import csv
write_path = os.getenv("WRITE_PATH")
print("write_path: ", write_path)

results_df_sorted = pd.read_csv(os.path.join(write_path, "model_evaluation_results.csv"), index_col=0
).sort_values(by="MAE_mean", ascending=True)

write_path_plots = os.getenv("WRITE_PATH_PLOTS")

if not os.path.exists(write_path_plots):
    os.makedirs(write_path_plots)
    print(f"Created plots directory: {write_path_plots}")

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(results_df_sorted))
ax.bar(
    x,
    results_df_sorted["MAE_mean"].to_numpy(),
    yerr=results_df_sorted["MAE_std"].to_numpy(),
    capsize=5,
    color=plt.cm.viridis(np.linspace(0.15, 0.85, len(x)))
)

ax.set_xticks(x)
ax.set_xticklabels(results_df_sorted.index, rotation=45)
ax.set_title("Model Comparison: Mean Absolute Error (MAE)")
ax.set_xlabel("Models")
ax.set_ylabel("Mean Absolute Error (MAE)")

fig.tight_layout()
fig.savefig(
    os.path.join(write_path_plots, "model_comparison_mae.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()


Bar plot of R² per model with baseline R² = 0

In [ ]:
#bar plot of R² per model with basel

ax = results_df_sorted["R2_mean"].plot.bar(
    yerr=results_df_sorted["R2_std"],
    capsize=5,
    color=plt.cm.viridis(np.linspace(0.15, 0.85, len(results_df_sorted))),
    figsize=(10, 6)
)

plt.axhline(0, color="red", linestyle="--", linewidth=1)
plt.title("Model Comparison: R² Score")
plt.xlabel("Models")
plt.ylabel("R² Score")

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    os.path.join(write_path_plots, "model_comparison_r2.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

Box plots of fold-level scores

In [ ]:
#box plots of fold scores for each model, for MAE, MSE, RMSE, R²

boxplot_metrics = ["MAE", "MSE", "RMSE", "R2"]

x = np.arange(len(results_df_sorted))
y = results_df_sorted[["MAE_mean", "MSE_mean", "RMSE_mean", "R2_mean"]].to_numpy()

plt.figure(figsize=(12, 8))
for i, metric in enumerate(boxplot_metrics):
    plt.subplot(2, 2, i + 1)
    sns.boxplot(
        x=results_df_sorted.index,
        y=y[:, i],
    )
    plt.title(f"Model Comparison: {metric}")
    plt.xlabel("Models")
    plt.ylabel(metric)
    plt.xticks(rotation=45)
    
plt.tight_layout()
plt.savefig(
    os.path.join(write_path_plots, "model_comparison_boxplots.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()



## SHAP training and evaluation

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    KFold
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
def train_models_with_shap_feature_counts(
    shap_model,
    evaluation_model,
    X,
    y,
    model_name,
    test_size=0.2,
    random_state=42,
    max_features=8
):
    """
    Rank features with SHAP using one model, then train and evaluate
    another model using the top 1 to max_features features.

    The SHAP model is used only for feature ranking.
    The evaluation model is re-tuned separately for every feature count.
    """

    if not isinstance(X, pd.DataFrame):
        raise TypeError("X must be a pandas DataFrame.")

    if not isinstance(y, pd.Series):
        y = pd.Series(y)

    if len(X) != len(y):
        raise ValueError("X and y must contain the same number of rows.")

    if X.isna().any().any():
        raise ValueError(
            "X contains missing values. Impute X before calling this function."
        )

    if max_features < 1 or max_features > X.shape[1]:
        raise ValueError(
            "max_features must be between 1 and the number of columns in X."
        )

    # Split once. The test data remains untouched until final evaluation.
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state
    )

    print(f"\nModel evaluated: {model_name}")
    print(f"Total observations: {len(X)}")
    print(f"Training observations: {len(X_train)}")
    print(f"Testing observations: {len(X_test)}")
    print(f"Available features: {X.shape[1]}")

    # ---------------------------------------------------------------
    # 1. Train a separate model for SHAP feature ranking
    # ---------------------------------------------------------------

    ranking_model = clone(shap_model)
    ranking_model.fit(X_train, y_train)

    # SHAP explains the complete model, including any preprocessing
    # contained inside a pipeline.
    masker = shap.maskers.Independent(
        X_train,
        max_samples=len(X_train)
    )

    explainer = shap.Explainer(
        ranking_model.predict,
        masker
    )

    shap_output = explainer(X_train)
    shap_values = shap_output.values

    if shap_values.ndim == 3:
        shap_values = np.mean(np.abs(shap_values), axis=2)

    mean_absolute_shap = np.abs(shap_values).mean(axis=0)

    shap_importance = (
        pd.Series(
            mean_absolute_shap,
            index=X_train.columns,
            name="mean_absolute_shap"
        )
        .sort_values(ascending=False)
    )

    print("\nSHAP feature ranking:")
    display(shap_importance.to_frame())

    ranked_features = shap_importance.index.tolist()

    # ---------------------------------------------------------------
    # 2. Train and evaluate models with the top 1 to 5 features
    # ---------------------------------------------------------------

    results = []
    trained_models = {}
    feature_sets = {}

    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state
    )

    for number_of_features in range(1, max_features + 1):

        selected_features = ranked_features[:number_of_features]

        print(
            f"\nTraining {model_name} with "
            f"the top {number_of_features} feature(s):"
        )
        print(selected_features)

        # Create a new model for this feature count.
        model_to_tune = clone(evaluation_model)

        # If the evaluation model is ElasticNet, tune its parameters
        # again because the optimal regularization changes when features
        # are removed.
        if isinstance(model_to_tune, ElasticNet):
            model_pipeline = make_pipeline(
                StandardScaler(),
                ElasticNet(
                    max_iter=10000,
                    random_state=random_state
                )
            )

            parameter_grid = {
                "elasticnet__alpha": np.logspace(-4, 1, 50),
                "elasticnet__l1_ratio": np.linspace(0.1, 1.0, 21)
            }

        else:
            model_pipeline = model_to_tune
            parameter_grid = {}

        if parameter_grid:
            search = GridSearchCV(
                estimator=model_pipeline,
                param_grid=parameter_grid,
                scoring="neg_mean_absolute_error",
                cv=cv,
                n_jobs=-1
            )

            search.fit(
                X_train[selected_features],
                y_train
            )

            model = search.best_estimator_
            best_parameters = search.best_params_

        else:
            model = model_pipeline
            model.fit(
                X_train[selected_features],
                y_train
            )
            best_parameters = {}

        predictions = model.predict(
            X_test[selected_features]
        )

        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test, predictions)

        number_of_nonzero_coefficients = None

        if hasattr(model, "named_steps"):
            final_estimator = list(model.named_steps.values())[-1]

            if hasattr(final_estimator, "coef_"):
                number_of_nonzero_coefficients = np.count_nonzero(
                    final_estimator.coef_
                )

        results.append({
            "model": model_name,
            "number_of_features": number_of_features,
            "features": selected_features,
            "best_parameters": best_parameters,
            "nonzero_coefficients": number_of_nonzero_coefficients,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

        trained_models[number_of_features] = model
        feature_sets[number_of_features] = selected_features

        print(f"Best parameters: {best_parameters}")
        print(f"Non-zero coefficients: {number_of_nonzero_coefficients}")
        print(f"MAE: {mae:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"R2: {r2:.4f}")

    results = pd.DataFrame(results)

    return (
        results,
        shap_importance,
        trained_models,
        feature_sets
    )

In [ ]:
df_final = pd.read_csv(os.path.join(write_path, "df_merged_lasso_selected_imputed.csv"))

X = df_final.drop(columns=[AFTER_MERGE_TARGET_VARIABLE])
y = df_final[AFTER_MERGE_TARGET_VARIABLE]

For ElasticNet, Random Forest is used for SHAP ranking because the previously saved ElasticNet model had all coefficients equal to zero. ElasticNet itself is still used for the final 1–5-feature models.

In [ ]:
elasticnet_results, elasticnet_shap_importance, \
elasticnet_trained_models, elasticnet_feature_sets = (
    train_models_with_shap_feature_counts(
        shap_model=best_randomforest_model,
        evaluation_model=ElasticNet(),
        X=X,
        y=y,
        model_name="ElasticNet",
        test_size=0.2,
        random_state=42,
        max_features=5
    )
)

#### Random Forest SHAP analysis

In [ ]:
best_randomforest_model = best_model_rf

randomforest_results, randomforest_shap_importance, \
randomforest_trained_models, randomforest_feature_sets = (
    train_models_with_shap_feature_counts(
        shap_model=best_randomforest_model,
        evaluation_model=best_randomforest_model,
        X=X,
        y=y,
        model_name="RandomForest",
        test_size=0.2,
        random_state=42,
        max_features=5
    )
)

display(randomforest_results)
display(randomforest_shap_importance.to_frame())

#### LightGBM SHAP analysis

In [ ]:
best_lightgbm_model = best_model_lgbm

lightgbm_results, lightgbm_shap_importance, \
lightgbm_trained_models, lightgbm_feature_sets = (
    train_models_with_shap_feature_counts(
        shap_model=best_lightgbm_model,
        evaluation_model=best_lightgbm_model,
        X=X,
        y=y,
        model_name="LightGBM",
        test_size=0.2,
        random_state=42,
        max_features=8
    )
)

display(lightgbm_results)
display(lightgbm_shap_importance.to_frame())

#### XGBoost SHAP analysis

In [ ]:
best_xgboost_model = best_model_xgb

xgboost_results, xgboost_shap_importance, \
xgboost_trained_models, xgboost_feature_sets = (
    train_models_with_shap_feature_counts(
        shap_model=best_xgboost_model,
        evaluation_model=best_xgboost_model,
        X=X,
        y=y,
        model_name="XGBoost",
        test_size=0.2,
        random_state=42,
        max_features=8
    )
)

display(xgboost_results)
display(xgboost_shap_importance.to_frame())

#### SVR SHAP analysis

In [ ]:
best_svr_model = best_model_svr

svr_results, svr_shap_importance, \
svr_trained_models, svr_feature_sets = (
    train_models_with_shap_feature_counts(
        shap_model=best_svr_model,
        evaluation_model=best_svr_model,
        X=X,
        y=y,
        model_name="SVR",
        test_size=0.2,
        random_state=42,
        max_features=8
    )
)

display(svr_results)
display(svr_shap_importance.to_frame())

### Combined SHAP analysis

In [ ]:
all_shap_results = pd.concat(
    [
        elasticnet_results,
        randomforest_results,
        lightgbm_results,
        xgboost_results,
        svr_results
    ],
    ignore_index=True
)

display(
    all_shap_results[
        [
            "model",
            "number_of_features",
            "features",
            "MAE",
            "RMSE",
            "R2"
        ]
    ]
)

In [ ]:
best_result_per_model = (
    all_shap_results
    .loc[
        all_shap_results.groupby("model")["MAE"].idxmin()
    ]
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(best_result_per_model)
#save best_result_per_model to csv in WRITE_PATH
best_result_per_model.to_csv(
    os.path.join(write_path, "best_result_per_model.csv"),
    index=False
)

### SHAP summary results

SHAP-based feature selection was applied to ElasticNet, Random Forest, LightGBM, XGBoost, and SVR.

For each model, features were ranked using their mean absolute SHAP values. New models were then trained with the top 1 to 5 features and evaluated using MAE, RMSE, and R².

The best feature count is selected according to the lowest test-set MAE. These results are exploratory because the same test set is used to compare the different feature counts. The nested cross-validation results remain the more reliable estimate of generalization performance.

A negative R² indicates that the model performs worse than a baseline model that always predicts the mean target value. Therefore, even when a small number of features produces the lowest MAE, this does not necessarily mean that the model has useful predictive performance.